<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Pytorch_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [44]:
#extracting data from csv using pandas
trans_data=pd.read_csv("/content/transactions.csv")
trans_data.head()

,transaction_amount,account_age_days,num_prior_transactions,avg_transaction_amount,is_fraud
0,5119.02,52,189,124.51,0
1,1124.91,210,173,301.45,0
2,4728.04,66,7,55.60,0
3,4047.79,55,143,95.56,1
4,1771.32,1207,71,327.58,0


In [45]:
trans_data.shape

(4000, 5)

In [46]:
trans_data.describe()

,transaction_amount,account_age_days,num_prior_transactions,avg_transaction_amount,is_fraud
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,3975.272090,770.622500,124.985500,211.833840,0.504500
std,2292.752411,435.403783,72.418795,110.704603,0.500042
min,13.710000,1.000000,0.000000,20.100000,0.000000
25%,1943.560000,386.000000,62.000000,114.827500,0.000000
50%,3959.985000,793.500000,125.000000,214.470000,1.000000
75%,5937.965000,1150.000000,188.000000,308.827500,1.000000
max,7998.070000,1500.000000,250.000000,399.960000,1.000000


In [47]:
trans_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_amount      4000 non-null   float64
 1   account_age_days        4000 non-null   int64  
 2   num_prior_transactions  4000 non-null   int64  
 3   avg_transaction_amount  4000 non-null   float64
 4   is_fraud                4000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 156.4 KB


In [48]:
trans_data.tail()

,transaction_amount,account_age_days,num_prior_transactions,avg_transaction_amount,is_fraud
3995,813.57,1258,209,370.51,1
3996,6181.68,962,10,218.66,0
3997,2944.60,1151,194,93.97,1
3998,4390.40,603,141,73.51,1
3999,1282.85,1243,117,318.27,0


In [49]:
#defining neural network
class fraud_classifier(nn.Module):
  def __init__(self,input_size,hidden_dims,dropput_p=0.3):
    super().__init__()
    layers=[]
    prev=input_size
    for h in hidden_dims:
      layers+=[nn.Linear(prev,h),nn.ReLU(),nn.Dropout(dropput_p)]
      prev=h # Corrected: 'prev' should be updated to 'h' (output dimension of current layer)
    layers.append(nn.Linear(prev,1))
    layers.append(nn.Sigmoid())
    self.net=nn.Sequential(*layers)

  def forward(self, x):
    return self.net(x)

In [50]:
#dataset preparation
raw_x=trans_data[["transaction_amount","account_age_days","num_prior_transactions","avg_transaction_amount"]].values
raw_y=trans_data[["is_fraud"]].values

In [51]:
raw_x

array([[5119.02,   52.  ,  189.  ,  124.51],
       [1124.91,  210.  ,  173.  ,  301.45],
       [4728.04,   66.  ,    7.  ,   55.6 ],
       ...,
       [2944.6 , 1151.  ,  194.  ,   93.97],
       [4390.4 ,  603.  ,  141.  ,   73.51],
       [1282.85, 1243.  ,  117.  ,  318.27]])

In [52]:
raw_y

array([[0],
       [0],
       [0],
       ...,
       [1],
       [1],
       [0]])

In [53]:
#splitting into the train and test
train_X,test_X,train_Y,test_Y=train_test_split(raw_x,raw_y,test_size=0.2,random_state=42)
scaler=StandardScaler()
train_X_np=scaler.fit_transform(train_X)
test_X_np=scaler.transform(test_X)

#converting into tensor
train_X_tensor=torch.tensor(train_X_np,dtype=torch.float32)
train_Y_tensor=torch.tensor(train_Y,dtype=torch.float32) # Changed dtype to float and removed .unsqueeze(1)
test_X_tensor=torch.tensor(test_X_np,dtype=torch.float32)
test_Y_tensor=torch.tensor(test_Y,dtype=torch.float32) # Changed dtype to float and removed .unsqueeze(1)

#creating dataset
train_dataset=TensorDataset(train_X_tensor,train_Y_tensor)
test_dataset=TensorDataset(test_X_tensor,test_Y_tensor)

In [59]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=fraud_classifier(input_size=4,hidden_dims=[16,8]).to(device)

In [57]:

def train_model(model, train_loader, test_loader, optimizer, epochs=50, print_every=10):
      criterion=nn.BCELoss()
      history={"train_loss":[],"test_loss":[],"test_accuracy":[]}
      for epoch in range(epochs):
        #---------training phase-------------
        total_loss=0.0
        for batch_x,batch_y in train_loader:
          optimizer.zero_grad()
          preds=model(batch_x.to(device))
          loss=criterion(preds,batch_y.to(device).float()) # Ensure batch_y is float
          loss.backward()
          optimizer.step()
          total_loss+=loss.item()
          # print(f"Epoch {epoch+1}/{epochs},--------- Loss: {total_loss:.4f}") # Removed verbose printing per batch
        total_loss/=len(train_loader)

        #------------Evaluation phase-------------
        model.eval()
        test_loss,correct=0.0,0
        with torch.no_grad():
          for batch_x,batch_y in test_loader:
            preds=model(batch_x.to(device))
            test_loss+=criterion(preds,batch_y.to(device).float()).item() # Ensure batch_y is float
            correct+=(preds.round()==batch_y.to(device)).sum().item()
        test_loss/=len(test_loader)
        accuracy=correct/len(test_loader.dataset)
        history["train_loss"].append(total_loss)
        history["test_loss"].append(test_loss)
        history["test_accuracy"].append(accuracy)

        if (epoch+1)%print_every==0:
          print(f"Epoch {epoch+1}/{epochs}, Train Loss: {total_loss:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.4f}")
      return history

In [62]:
optimizer=optim.SGD(model.parameters(),lr=0.01,momentum=0.9)
print("Testing with Stochastic gradient descent with momentum 0.9")
history=train_model(model,train_loader,test_loader,optimizer,epochs=100)

Testing with Stochastic gradient descent with momentum 0.9
Epoch 10/100, Train Loss: 0.6846, Test Loss: 0.7037, Test Accuracy: 0.4925
Epoch 20/100, Train Loss: 0.6844, Test Loss: 0.7049, Test Accuracy: 0.4913
Epoch 30/100, Train Loss: 0.6824, Test Loss: 0.7079, Test Accuracy: 0.4975
Epoch 40/100, Train Loss: 0.6821, Test Loss: 0.7025, Test Accuracy: 0.4900
Epoch 50/100, Train Loss: 0.6801, Test Loss: 0.7055, Test Accuracy: 0.4900
Epoch 60/100, Train Loss: 0.6809, Test Loss: 0.7035, Test Accuracy: 0.5062
Epoch 70/100, Train Loss: 0.6812, Test Loss: 0.7060, Test Accuracy: 0.4813
Epoch 80/100, Train Loss: 0.6813, Test Loss: 0.7041, Test Accuracy: 0.4963
Epoch 90/100, Train Loss: 0.6794, Test Loss: 0.7081, Test Accuracy: 0.4888
Epoch 100/100, Train Loss: 0.6797, Test Loss: 0.7074, Test Accuracy: 0.4963


In [63]:
optimizer=optim.Adam(model.parameters(),lr=0.001)
print("Testing with Adam")
history=train_model(model,train_loader,test_loader,optimizer,epochs=100)

Testing with Stochastic gradient descent with momentum 0.9
Epoch 10/100, Train Loss: 0.6743, Test Loss: 0.7082, Test Accuracy: 0.4963
Epoch 20/100, Train Loss: 0.6740, Test Loss: 0.7079, Test Accuracy: 0.4938
Epoch 30/100, Train Loss: 0.6736, Test Loss: 0.7080, Test Accuracy: 0.5025
Epoch 40/100, Train Loss: 0.6722, Test Loss: 0.7083, Test Accuracy: 0.5050
Epoch 50/100, Train Loss: 0.6720, Test Loss: 0.7089, Test Accuracy: 0.4950
Epoch 60/100, Train Loss: 0.6718, Test Loss: 0.7098, Test Accuracy: 0.4988
Epoch 70/100, Train Loss: 0.6708, Test Loss: 0.7101, Test Accuracy: 0.4988
Epoch 80/100, Train Loss: 0.6704, Test Loss: 0.7100, Test Accuracy: 0.5100
Epoch 90/100, Train Loss: 0.6693, Test Loss: 0.7090, Test Accuracy: 0.5112
Epoch 100/100, Train Loss: 0.6691, Test Loss: 0.7107, Test Accuracy: 0.5062
